# CIFAR-10 CNN for object classification on the e-puck (Webots)

This notebook trains a convolutional neural network (CNN) on CIFAR-10. It also:

1. compares the SGD and Adam optimisers under identical conditions,
2. evaluates the final model on the held-out test set (accuracy, confusion matrix, most-confused classes),
3. classifies images of our own,
4. prepares the model for the robot: TensorFlow Lite conversion, int8 quantisation and inference-time measurement.

The saved model (`final_model.keras`) is loaded by the Webots controller `opencv.py`.

## About the dataset

CIFAR-10 is a labelled subset of the *80 Million Tiny Images* collection. It contains **60,000 colour images of 32 × 32 pixels in 10 classes** (6,000 per class): **50,000 for training** and **10,000 for testing**.

Classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

To keep the test set unseen until the final evaluation, 5,000 training images are held out as a **validation set** for monitoring training and early stopping.

## Running this notebook in Google Colab

1. Open it from the repository with the **Open in Colab** button (or **File → Upload notebook** at [colab.research.google.com](https://colab.research.google.com)).
2. **Runtime → Change runtime type → T4 GPU**. Training then takes a few minutes instead of over an hour on a laptop CPU.
3. **Runtime → Run all**. Our own images are downloaded from the project repository automatically; to try your own photos, upload them into the `my_images` folder (Files panel) and re-run that cell.
4. At the end the trained models are downloaded. Copy `final_model.keras` into the Webots controller folder `controllers/opencv/`.

The cells marked *Colab only* do nothing when the notebook runs in local Jupyter.

In [ ]:
# Colab only: report whether a GPU is available (training works without one, just slower)
import sys
if 'google.colab' in sys.modules:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print("GPU:", gpus[0].name if gpus else "none - use Runtime > Change runtime type > T4 GPU")

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

SEED = 42
tf.keras.utils.set_random_seed(SEED)   # seeds Python, NumPy and TensorFlow
print("TensorFlow", tf.__version__)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Labels come as shape (N, 1); flatten them to (N,)
train_labels = train_labels.flatten()
test_labels = test_labels.flatten()

# Normalise pixel values to [0, 1]
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0

# Hold out the last 5,000 training images for validation (the test set stays unseen)
x_train, y_train = train_images[:45000], train_labels[:45000]
x_val, y_val = train_images[45000:], train_labels[45000:]

print(f"train {x_train.shape}, validation {x_val.shape}, test {test_images.shape}")

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i]])
plt.suptitle('CIFAR-10 training samples')
plt.tight_layout()
plt.show()

## Input pipeline and data augmentation

Random horizontal flips and small shifts (up to 10 %) are applied **to the training images only**. The augmentation runs inside the `tf.data` pipeline rather than in the model, so the saved model contains only the classifier and the robot controller can use it directly.

In [ ]:
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomTranslation(0.1, 0.1, fill_mode='reflect'),
], name='augmentation')

def make_dataset(x, y, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(10000, seed=SEED)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda img, lbl: (augment(img, training=True), lbl),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val, training=False)
test_ds = make_dataset(test_images, test_labels, training=False)

## Creating the CNN model

Two convolutional blocks, each with two 3 × 3 convolutions, batch normalisation, ReLU and 2 × 2 max pooling. A dense classifier with dropout follows. The last layer uses **softmax**, so the model outputs class probabilities directly; these are used as the confidence score.

In [ ]:
def conv_bn_relu(filters):
    return [layers.Conv2D(filters, (3, 3), padding='same', use_bias=False),
            layers.BatchNormalization(),
            layers.Activation('relu')]

DROPOUT = 0.2   # 0.5 stopped Adam from learning in an earlier run; 0.2 trains well with both optimisers

def build_model():
    return models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        *conv_bn_relu(32),
        *conv_bn_relu(32),
        layers.MaxPooling2D((2, 2)),
        *conv_bn_relu(64),
        *conv_bn_relu(64),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(DROPOUT),
        layers.Dense(10, activation='softmax'),
    ], name='cifar10_cnn')

build_model().summary()

## Training: SGD vs Adam

Each optimiser trains a **freshly initialised** model with the same data, epochs and seed, so the comparison is fair. Early stopping monitors the validation accuracy and restores the best weights.

In [ ]:
EPOCHS = 30

def train(optimizer, label):
    tf.keras.utils.set_random_seed(SEED)
    model = build_model()
    model.compile(optimizer=optimizer,
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  metrics=['accuracy'])
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True)
    start = time.time()
    history = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds,
                        callbacks=[early_stop], verbose=2)
    print(f"{label}: trained in {(time.time() - start) / 60:.1f} min")
    return model, history

sgd_model, sgd_history = train(tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9), 'SGD')

In [ ]:
adam_model, adam_history = train(tf.keras.optimizers.Adam(learning_rate=1e-3), 'Adam')

In [ ]:
def plot_history(histories):
    fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))
    for label, h in histories.items():
        ax_acc.plot(h.history['accuracy'], label=f'{label} train')
        ax_acc.plot(h.history['val_accuracy'], '--', label=f'{label} validation')
        ax_loss.plot(h.history['loss'], label=f'{label} train')
        ax_loss.plot(h.history['val_loss'], '--', label=f'{label} validation')
    ax_acc.set(xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1), title='Accuracy')
    ax_loss.set(xlabel='Epoch', ylabel='Loss', title='Loss')
    for ax in (ax_acc, ax_loss):
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

plot_history({'SGD': sgd_history, 'Adam': adam_history})

results = {}
for label, m in [('SGD', sgd_model), ('Adam', adam_model)]:
    loss, acc = m.evaluate(test_ds, verbose=0)
    results[label] = acc
    print(f"{label:4s} test accuracy: {acc:.2%}  (test loss {loss:.3f})")

## Final model evaluation

The optimiser with the higher test accuracy gives the final model.

In [ ]:
best_label = max(results, key=results.get)
model = adam_model if best_label == 'Adam' else sgd_model
print(f"Final model: {best_label}, test accuracy {results[best_label]:.2%}")

probs = model.predict(test_ds, verbose=0)
pred = probs.argmax(axis=1)

# Confusion matrix (rows = true class, columns = predicted class)
cm = tf.math.confusion_matrix(test_labels, pred, num_classes=10).numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10), class_names, rotation=45, ha='right')
ax.set_yticks(range(10), class_names)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('Confusion matrix (CIFAR-10 test set)')
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Per-class accuracy:")
for i, name in enumerate(class_names):
    print(f"  {name:10s} {cm[i, i] / cm[i].sum():.1%}")

off_diag = [(cm[i, j], class_names[i], class_names[j])
            for i in range(10) for j in range(10) if i != j]
print("\nMost frequent confusions (true -> predicted):")
for count, t, p in sorted(off_diag, reverse=True)[:5]:
    print(f"  {t:10s} -> {p:10s} {count} images")

In [ ]:
model.save('final_model.keras')
print("Saved final_model.keras")

## Predicting images of our own

Put photos in the `my_images/` folder (any size; they are resized to 32 × 32). Each one is shown with its predicted class and confidence.

In [ ]:
# Colab only: fetch our own images from the project repository (skipped in local Jupyter)
import os, sys, urllib.request
if 'google.colab' in sys.modules:
    os.makedirs('my_images', exist_ok=True)
    base = 'https://raw.githubusercontent.com/sinegan-ux/epuck-cnn-part2/main/notebooks/my_images/'
    for name in ['photo_car.jpg', 'photo_horse.jpg', 'photo_cat.jpg', 'webots_car.png', 'webots_horse.png']:
        if not os.path.exists(os.path.join('my_images', name)):
            urllib.request.urlretrieve(base + name, os.path.join('my_images', name))
    print("Images:", ", ".join(sorted(os.listdir('my_images'))))

In [ ]:
def load_image(filename):
    # Load an image file as a 1x32x32x3 float32 batch in [0, 1]
    img = tf.keras.utils.load_img(filename, target_size=(32, 32))
    img = tf.keras.utils.img_to_array(img) / 255.0
    return img[np.newaxis, ...].astype('float32')

def classify(model, filename):
    p = model.predict(load_image(filename), verbose=0)[0]
    idx = int(np.argmax(p))
    return class_names[idx], float(p[idx])

saved_model = tf.keras.models.load_model('final_model.keras')

IMAGE_DIR = 'my_images'
files = sorted(f for f in os.listdir(IMAGE_DIR)
               if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))) if os.path.isdir(IMAGE_DIR) else []

if not files:
    print(f"No images found in '{IMAGE_DIR}/'. Add some photos and re-run this cell.")
else:
    fig, axes = plt.subplots(1, len(files), figsize=(3.2 * len(files), 3.6), squeeze=False)
    for ax, f in zip(axes[0], files):
        path = os.path.join(IMAGE_DIR, f)
        name, conf = classify(saved_model, path)
        print(f"{f}: {name} ({conf:.1%})")
        ax.imshow(tf.keras.utils.load_img(path))
        ax.set_title(f"{name} ({conf:.0%})")
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Optimisation for the robot: TensorFlow Lite and int8 quantisation

The controller must keep the e-puck moving, so inference has to be cheap. The model is converted to TensorFlow Lite in two forms: plain float32, and **full-integer (int8) quantisation**, calibrated on 500 training images. Model size, test accuracy and single-image inference time are then compared.

In [ ]:
def representative_data():
    for i in range(500):
        yield [x_train[i:i + 1]]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_float = converter.convert()

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_int8 = converter.convert()

with open('final_model_int8.tflite', 'wb') as f:
    f.write(tflite_int8)

In [ ]:
class TFLiteClassifier:
    def __init__(self, model_bytes):
        self.interp = tf.lite.Interpreter(model_content=model_bytes)
        self.interp.allocate_tensors()
        self.inp = self.interp.get_input_details()[0]
        self.out = self.interp.get_output_details()[0]

    def __call__(self, x):  # x: 1x32x32x3 float32 in [0, 1]
        if self.inp['dtype'] == np.uint8:
            scale, zero = self.inp['quantization']
            x = np.clip(np.round(x / scale + zero), 0, 255).astype(np.uint8)
        self.interp.set_tensor(self.inp['index'], x)
        self.interp.invoke()
        return self.interp.get_tensor(self.out['index'])[0]

def tflite_accuracy(clf):
    preds = [np.argmax(clf(test_images[i:i + 1])) for i in range(len(test_images))]
    return np.mean(np.array(preds) == test_labels)

def time_per_image(fn, runs=300):
    x = test_images[:1]
    for _ in range(20):          # warm-up
        fn(x)
    start = time.perf_counter()
    for _ in range(runs):
        fn(x)
    return (time.perf_counter() - start) / runs * 1000  # ms

model.save('tmp_size_check.keras')
keras_size = os.path.getsize('tmp_size_check.keras') / 1024
os.remove('tmp_size_check.keras')

float_clf, int8_clf = TFLiteClassifier(tflite_float), TFLiteClassifier(tflite_int8)

rows = [
    ('Keras model.predict()', keras_size, results[best_label],
     time_per_image(lambda x: model.predict(x, verbose=0), runs=100)),
    ('Keras model(x) call', keras_size, results[best_label],
     time_per_image(lambda x: model(x, training=False))),
    ('TFLite float32', len(tflite_float) / 1024, tflite_accuracy(float_clf),
     time_per_image(float_clf)),
    ('TFLite int8', len(tflite_int8) / 1024, tflite_accuracy(int8_clf),
     time_per_image(int8_clf)),
]

print(f"{'Version':24s}{'Size (KB)':>11s}{'Test acc.':>11s}{'ms / image':>12s}")
for name, size, acc, ms in rows:
    print(f"{name:24s}{size:11.0f}{acc:11.2%}{ms:12.2f}")

In [ ]:
# Colab only: download the trained models to your computer (skipped in local Jupyter)
import sys
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('final_model.keras')
    files.download('final_model_int8.tflite')

## Summary

The test accuracy, per-class accuracy, most-confused class pairs, own-image predictions and TensorFlow Lite timing printed above are the figures used in the presentation.